In [ ]:
{
    "cells": [
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "# ADUserReview.ipynb\n",
                "**Purpose**: Review AD Group and GPO data and tag decisions interactively using dropdowns and widgets.\n"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": null,
            "metadata": {},
            "outputs": [],
            "source": [
                "# Load AD Discovery JSON\n",
                "import json\n",
                "import pandas as pd\n",
                "from pathlib import Path\n",
                "\n",
                "json_path = Path(\"exports/adminpdavidson/ad_user_summary_20250528_151441.json\")\n",
                "with open(json_path, 'r', encoding='utf-8') as f:\n",
                "    ad_data = json.load(f)"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": null,
            "metadata": {},
            "outputs": [],
            "source": [
                "# Extract Groups and GPOs\n",
                "groups = ad_data.get(\"Groups\", [])\n",
                "gpos = ad_data.get(\"GPOs\", [])\n",
                "\n",
                "df_groups = pd.DataFrame(groups, columns=[\"GroupName\"])\n",
                "df_groups[\"Decision\"] = \"\"\n",
                "\n",
                "df_gpos = pd.DataFrame(gpos, columns=[\"GPOName\"])\n",
                "df_gpos[\"Decision\"] = \"\""
            ]
        },
        {
            "cell_type": "code",
            "execution_count": null,
            "metadata": {},
            "outputs": [],
            "source": [
                "# Interactive Dropdown for Groups\n",
                "import ipywidgets as widgets\n",
                "from IPython.display import display, Markdown\n",
                "\n",
                "group_widgets = []\n",
                "group_decisions = {}\n",
                "options = ['Keep', 'Remove', 'CAB Review', 'Unsure']\n",
                "\n",
                "display(Markdown(\"### Group Membership Review – select a decision per group\"))\n",
                "\n",
                "for group in df_groups['GroupName']:\n",
                "    dropdown = widgets.Dropdown(\n",
                "        options=options,\n",
                "        value='Unsure',\n",
                "        description=group[:30] + ('' if len(group) < 30 else '...'),\n",
                "        style={'description_width': 'initial'},\n",
                "        layout=widgets.Layout(width='80%')\n",
                "    )\n",
                "    group_widgets.append(dropdown)\n",
                "    group_decisions[group] = dropdown\n",
                "    display(dropdown)"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": null,
            "metadata": {},
            "outputs": [],
            "source": [
                "# Save Group Decisions\n",
                "def save_group_decisions(button):\n",
                "    for group, dropdown in group_decisions.items():\n",
                "        df_groups.loc[df_groups['GroupName'] == group, 'Decision'] = dropdown.value\n",
                "    df_groups.to_csv(\"interactive_decision_groups.csv\", index=False)\n",
                "    print(\"[INFO] Group decisions saved: interactive_decision_groups.csv\")\n",
                "\n",
                "save_button = widgets.Button(description=\"Save Group Decisions\")\n",
                "save_button.on_click(save_group_decisions)\n",
                "display(save_button)"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": null,
            "metadata": {},
            "outputs": [],
            "source": [
                "# Interactive Dropdown for GPOs\n",
                "gpo_widgets = []\n",
                "gpo_decisions = {}\n",
                "\n",
                "display(Markdown(\"### GPO Linkage Review – select a decision per GPO\"))\n",
                "\n",
                "for gpo in df_gpos['GPOName']:\n",
                "    dropdown = widgets.Dropdown(\n",
                "        options=options,\n",
                "        value='Unsure',\n",
                "        description=gpo[:30] + ('' if len(gpo) < 30 else '...'),\n",
                "        style={'description_width': 'initial'},\n",
                "        layout=widgets.Layout(width='80%')\n",
                "    )\n",
                "    gpo_widgets.append(dropdown)\n",
                "    gpo_decisions[gpo] = dropdown\n",
                "    display(dropdown)"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": null,
            "metadata": {},
            "outputs": [],
            "source": [
                "# Save GPO Decisions\n",
                "def save_gpo_decisions(button):\n",
                "    for gpo, dropdown in gpo_decisions.items():\n",
                "        df_gpos.loc[df_gpos['GPOName'] == gpo, 'Decision'] = dropdown.value\n",
                "    df_gpos.to_csv(\"interactive_decision_gpos.csv\", index=False)\n",
                "    print(\"[INFO] GPO decisions saved: interactive_decision_gpos.csv\")\n",
                "\n",
                "save_button_gpo = widgets.Button(description=\"Save GPO Decisions\")\n",
                "save_button_gpo.on_click(save_gpo_decisions)\n",
                "display(save_button_gpo)"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": null,
            "metadata": {},
            "outputs": [],
            "source": [
                "# Export Markdown Summary\n",
                "def export_markdown_summary():\n",
                "    lines = [\"# AD Review Summary\\n\"]\n",
                "    lines.append(\"## Group Membership Decisions\\n\")\n",
                "    for _, row in df_groups.iterrows():\n",
                "        lines.append(f\"- **{row['GroupName']}**: {row['Decision']}\")\n",
                "\n",
                "    lines.append(\"\\n## GPO Linkage Decisions\\n\")\n",
                "    for _, row in df_gpos.iterrows():\n",
                "        lines.append(f\"- **{row['GPOName']}**: {row['Decision']}\")\n",
                "\n",
                "    with open(\"decision_summary.md\", \"w\", encoding=\"utf-8\") as f:\n",
                "        f.write(\"\\n\".join(lines))\n",
                "    print(\"[INFO] Markdown summary saved: decision_summary.md\")\n",
                "\n",
                "export_button = widgets.Button(description=\"Export Markdown Summary\")\n",
                "export_button.on_click(lambda b: export_markdown_summary())\n",
                "display(export_button)"
            ]
        }
    ],
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "name": "python",
            "version": "3.11"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 5
}